# 1: Install libraries



In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece sacrebleu rouge_score bert_score nltk pandas scikit-learn openpyxl

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.7 MB/s eta 0:00:00


# 2: Import libraries

In [ ]:
import os
import re
import gc
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    EncoderDecoderModel,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


GPU available: True
GPU: Tesla T4


# 3: Upload dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

print("Uploaded file:", csv_path)

Saving Counterspeech_dataset_Old_cleaned (1).csv to Counterspeech_dataset_Old_cleaned (1).csv
Uploaded file: Counterspeech_dataset_Old_cleaned (1).csv


# 4: Load dataset

In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (3011, 5)
Columns: ['id', 'offensive_text', 'counterspeech_text', 'counterspeech_type', 'text_type']


,id,offensive_text,counterspeech_text,counterspeech_type,text_type
0,1,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,fact_based,hate_speech
1,2,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...,warning_of_consequences,sexual_harassment
2,3,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।,empathy,hate_speech
3,4,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...,fact_based,gender_abuse
4,5,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।,moral_appeal,abusive_language


# 5: Keep only generation columns

In [ ]:
required_columns = ["offensive_text", "counterspeech_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df[required_columns].copy()

df.head()

,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 6: Clean text

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    return text

df["offensive_text"] = df["offensive_text"].apply(clean_text)
df["counterspeech_text"] = df["counterspeech_text"].apply(clean_text)

df = df.dropna()
df = df.drop_duplicates()
df = df[df["offensive_text"].str.len() > 2]
df = df[df["counterspeech_text"].str.len() > 2]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (3011, 2)


,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 7: Create input and target text

In [ ]:
# Do not prepend a long identical instruction to every input.
# The offensive text itself is the encoder input.
df["input_text"] = df["offensive_text"]
df["target_text"] = df["counterspeech_text"]

df = df[["input_text", "target_text"]]

df.head()


,input_text,target_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 8: Train-validation-test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))
print("Test size:", len(test_df))

Train size: 2408
Validation size: 301
Test size: 302


# 9: Convert to Hugging Face Dataset

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

dataset

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 302
    })
})

# 10: Load BanglaBERT Encoder--Decoder

> **Important:** `csebuetnlp/banglabert` is an encoder-only ELECTRA discriminator and cannot generate counterspeech by itself.  
> This notebook constructs a generative BanglaBERT-family model using:
>
> - Encoder: `csebuetnlp/banglabert`
> - Decoder: `csebuetnlp/banglabert_generator`
>
> The resulting system should be reported as **BanglaBERT Encoder--Decoder**, not as the original standalone BanglaBERT model.


In [ ]:
ENCODER_NAME = "csebuetnlp/banglabert"
DECODER_NAME = "csebuetnlp/banglabert_generator"

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

# BanglaBERT's tokenizer uses CLS/SEP rather than BOS/EOS.
# Set aliases before creating Trainer so Transformers does not overwrite EOS with None.
tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token

model = EncoderDecoderModel.from_encoder_decoder_pretrained(
    ENCODER_NAME,
    DECODER_NAME
)

# Explicit decoder configuration.
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.bos_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

model.decoder.config.is_decoder = True
model.decoder.config.add_cross_attention = True
model.decoder.config.use_cache = False
model.config.use_cache = False

# Keep generation config synchronized.
model.generation_config.decoder_start_token_id = tokenizer.cls_token_id
model.generation_config.bos_token_id = tokenizer.cls_token_id
model.generation_config.eos_token_id = tokenizer.sep_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id

model.to(device)

print("Encoder:", ENCODER_NAME)
print("Decoder:", DECODER_NAME)
print("Device:", device)
print("Tokenizer vocab size:", len(tokenizer))
print("Decoder vocab size:", model.config.decoder.vocab_size)
print("BOS / decoder start:", tokenizer.bos_token, tokenizer.cls_token_id)
print("EOS:", tokenizer.eos_token, tokenizer.sep_token_id)
print("PAD:", tokenizer.pad_token, tokenizer.pad_token_id)


config.json:   0%|          | 0.00/586 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/528k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  443MB            

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  140MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/205 [00:00<?, ?it/s]

[transformers] ElectraForCausalLM LOAD REPORT from: csebuetnlp/banglabert_generator
Key                                                                   | Status  | 
----------------------------------------------------------------------+---------+-
electra.encoder.layer.{0...11}.crossattention.output.dense.weight     | MISSING | 
electra.encoder.layer.{0...11}.crossattention.self.key.weight         | MISSING | 
electra.encoder.layer.{0...11}.crossattention.self.key.bias           | MISSING | 
electra.encoder.layer.{0...11}.crossattention.output.LayerNorm.bias   | MISSING | 
electra.encoder.layer.{0...11}.crossattention.output.LayerNorm.weight | MISSING | 
electra.encoder.layer.{0...11}.crossattention.self.query.bias         | MISSING | 
electra.encoder.layer.{0...11}.crossattention.self.query.weight       | MISSING | 
electra.encoder.layer.{0...11}.crossattention.output.dense.bias       | MISSING | 
electra.encoder.layer.{0...11}.crossattention.self.value.bias         | MISSING | 
ele

Encoder: csebuetnlp/banglabert
Decoder: csebuetnlp/banglabert_generator
Device: cuda
Tokenizer vocab size: 32000
Decoder vocab size: 32000
BOS / decoder start: [CLS] 2
EOS: [SEP] 3
PAD: [PAD] 0


# 11: Tokenize dataset

In [ ]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 96

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True,
        add_special_tokens=True
    )

    all_labels = []

    for target in examples["target_text"]:
        # Do not add CLS to the label sequence because the decoder start token
        # is already CLS. Append SEP manually as the end token.
        target_ids = tokenizer(
            str(target),
            max_length=MAX_TARGET_LENGTH - 1,
            truncation=True,
            padding=False,
            add_special_tokens=False
        )["input_ids"]

        target_ids = target_ids + [tokenizer.sep_token_id]

        if len(target_ids) < MAX_TARGET_LENGTH:
            target_ids += [tokenizer.pad_token_id] * (
                MAX_TARGET_LENGTH - len(target_ids)
            )
        else:
            target_ids = target_ids[:MAX_TARGET_LENGTH]
            target_ids[-1] = tokenizer.sep_token_id

        labels = [
            token_id if token_id != tokenizer.pad_token_id else -100
            for token_id in target_ids
        ]
        all_labels.append(labels)

    model_inputs["labels"] = all_labels
    return model_inputs


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

tokenized_dataset


Map:   0%|          | 0/2408 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  140MB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Map:   0%|          | 0/302 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 302
    })
})

# 12: Debug tokenization

In [ ]:
sample = tokenized_dataset["train"][0]

decoded_input = tokenizer.decode(
    [x for x in sample["input_ids"] if x != tokenizer.pad_token_id],
    skip_special_tokens=True
)

valid_label_ids = [x for x in sample["labels"] if x != -100]

decoded_label = tokenizer.decode(
    valid_label_ids,
    skip_special_tokens=True
)

print("Decoded input:")
print(decoded_input)

print("\nDecoded target:")
print(decoded_label)

print("\nFirst valid label ID:", valid_label_ids[0])
print("Expected: not CLS =", tokenizer.cls_token_id)
print("Last valid label ID:", valid_label_ids[-1])
print("Expected SEP =", tokenizer.sep_token_id)
print("Valid target tokens:", len(valid_label_ids))


Decoded input:
নারীরা যাচ্ছে, কিন্তু সেটা বললে হুরেরা এসে আমার চৌদ্দগুষ্টি তুলে গালাগালি করবে ।

Decoded target:
ভিন্ন মতের মানুষকে না করে আলোচনা করুন । অসম্মানজনক ভাষা বিতর্ক অনেক বেশি ফলপ্রসূ.

First valid label ID: 3442
Expected: not CLS = 2
Last valid label ID: 3
Expected SEP = 3
Valid target tokens: 20


# 13: Data collator

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)


# 14: Batch loss test

In [ ]:
from torch.utils.data import DataLoader
import math

small_train_dataset = tokenized_dataset["train"].select(
    range(min(4, len(tokenized_dataset["train"])))
)

test_loader = DataLoader(
    small_train_dataset,
    batch_size=1,
    collate_fn=data_collator
)

batch = next(iter(test_loader))

print("Input shape:", tuple(batch["input_ids"].shape))
print("Label shape:", tuple(batch["labels"].shape))
print("Valid label tokens:", int((batch["labels"] != -100).sum()))

batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

test_loss = float(outputs.loss.item())
print("Test batch loss:", test_loss)

if not math.isfinite(test_loss) or test_loss <= 0:
    raise ValueError(
        "Invalid batch loss. Do not start training until tokenization and labels are fixed."
    )


Input shape: (1, 128)
Label shape: (1, 96)
Valid label tokens: 20
Test batch loss: 23.089447021484375


/usr/local/lib/python3.12/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:444: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


# 15: Training arguments

In [ ]:
OUTPUT_DIR = "/content/banglabert_encoder_decoder_counterspeech_v2"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    # A custom optimizer in Cell 32 uses separate encoder/decoder LRs.
    learning_rate=1e-4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    num_train_epochs=8,
    weight_decay=0.01,
    warmup_ratio=0.10,
    max_grad_norm=1.0,
    label_smoothing_factor=0.05,

    predict_with_generate=False,
    fp16=False,

    logging_steps=25,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=SEED
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# 16: Create trainer

In [ ]:
from torch.optim import AdamW

# Cross-attention in the decoder is newly initialized, so train the decoder
# faster while updating the pretrained encoder conservatively.
optimizer = AdamW(
    [
        {
            "params": [p for p in model.encoder.parameters() if p.requires_grad],
            "lr": 1e-5
        },
        {
            "params": [p for p in model.decoder.parameters() if p.requires_grad],
            "lr": 1e-4
        }
    ],
    weight_decay=0.01
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
    ],

    optimizers=(optimizer, None)
)


# 17: Train BanglaBERT Encoder--Decoder


In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,34.360967,3.794027
2,27.927935,3.331698
3,25.878215,3.185150
4,24.152292,3.112637
5,22.213738,3.068103
6,22.139556,3.052035
7,21.033145,3.041151
8,20.186780,3.046855


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['decoder.generator_lm_head.weight'].


TrainOutput(global_step=1208, training_loss=28.232101263589417, metrics={'train_runtime': 1253.623, 'train_samples_per_second': 15.367, 'train_steps_per_second': 0.964, 'total_flos': 1454671147302912.0, 'train_loss': 28.232101263589417, 'epoch': 8.0})

# 18: Evaluate loss

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
20.186780,3.041151,8


{'eval_loss': 3.041151285171509}

# 19: Save model

In [ ]:
FINAL_MODEL_DIR = "/content/final_banglabert_encoder_decoder_counterspeech_v2"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:", FINAL_MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/final_banglabert_encoder_decoder_counterspeech_v2


# 20: Manual generation test

In [ ]:
model.eval()

def clean_generated_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def generate_counterspeech(text, use_sampling=True):
    inputs = tokenizer(
        str(text),
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    generation_args = {
        "input_ids": inputs["input_ids"],
        "attention_mask": inputs["attention_mask"],
        "max_new_tokens": 55,
        "min_new_tokens": 5,
        "repetition_penalty": 1.25,
        "no_repeat_ngram_size": 3,
        "decoder_start_token_id": tokenizer.cls_token_id,
        "eos_token_id": tokenizer.sep_token_id,
        "pad_token_id": tokenizer.pad_token_id,
        "remove_invalid_values": True,
        "renormalize_logits": True
    }

    if use_sampling:
        generation_args.update({
            "do_sample": True,
            "temperature": 0.85,
            "top_k": 50,
            "top_p": 0.92,
            "num_beams": 1
        })
    else:
        generation_args.update({
            "do_sample": False,
            "num_beams": 4,
            "early_stopping": True
        })

    with torch.no_grad():
        output_ids = model.generate(**generation_args)

    output_text = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

    return clean_generated_text(output_text)


preview_predictions = []

for i in range(min(20, len(test_df))):
    text = test_df["input_text"].iloc[i]
    reference = test_df["target_text"].iloc[i]
    prediction = generate_counterspeech(text, use_sampling=True)
    preview_predictions.append(prediction)

    print("Input:", text)
    print("Reference:", reference)
    print("Generated:", prediction)
    print("Generated length:", len(prediction.split()))
    print("-" * 80)

unique_count = len(set(preview_predictions))
unique_rate = unique_count / len(preview_predictions)

print("Unique preview outputs:", unique_count, "/", len(preview_predictions))
print("Unique preview rate:", unique_rate)

if unique_rate < 0.50:
    raise RuntimeError(
        "Generation is still mode-collapsed. Do not calculate or save final metrics."
    )


Input: কি মনে করছ, মেয়েরা কিছু বোঝে ?
Reference: এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথে কাজ করছে।
Generated: কারও ধর্ম বিদ্রূপ না করে নির্দিষ্ট আচরণ নিয়ে কথা বলুন ।
Generated length: 11
--------------------------------------------------------------------------------
Input: আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক মূল জিনিস এড়িয়ে যাস!
Reference: প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আচরণ উল্লেখ করলে কথা শক্ত হয়।
Generated: কারও জন্ম এমন অপমানজনক কথা খুবই অশোভন ।
Generated length: 8
--------------------------------------------------------------------------------
Input: নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।
Reference: মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ভদ্র ভাষাই ভালো।
Generated: কারও জন্ম বা পরিবারের ওপর অপমান করা খুবই ।
Generated length: 9
--------------------------------------------------------------------------------
Input: ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি
Reference: একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।
Generated: কারও ধর্ম ব

# 21: Generate full test predictions

In [ ]:
test_inputs = test_df["input_text"].tolist()
references = test_df["target_text"].tolist()

predictions = []

for i, text in enumerate(test_inputs):
    pred = generate_counterspeech(text, use_sampling=True)
    predictions.append(pred)

    if i % 50 == 0:
        print(f"Generated {i}/{len(test_inputs)}")

results_df = pd.DataFrame({
    "input_text": test_inputs,
    "reference_counterspeech": references,
    "generated_counterspeech": predictions
})

results_df["generated_length"] = results_df[
    "generated_counterspeech"
].apply(lambda x: len(str(x).split()))

unique_output_rate = (
    results_df["generated_counterspeech"].nunique() / len(results_df)
)

print("Unique output rate:", unique_output_rate)

if unique_output_rate < 0.50:
    raise RuntimeError(
        "Too many duplicate generations. Do not use these results as final metrics."
    )

results_df.head(10)


Generated 0/302
Generated 50/302
Generated 100/302
Generated 150/302
Generated 200/302
Generated 250/302
Generated 300/302
Unique output rate: 0.9834437086092715


,input_text,reference_counterspeech,generated_counterspeech,generated_length
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,কারও পোশাক বা চেহারা এমন অপমান করা খুবই নিচু ম...,12
1,আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক ম...,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,কোনো গোষ্ঠীর বিরুদ্ধে এমন গালি না দিয়ে নির্দি...,12
2,"নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।","মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",এভাবে কাউকে খারাপ দেখালে সমস্যার সমাধান হয় না ।,9
3,ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,কাউকে লুচ্চা বলা বা পাগল বলে আক্রমণ না করে আচর...,13
4,নাস্তিক ব্লগারদের সাথে তর্ক বিতর্ক করে লাভ নাই...,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,ধর্ম কথা বলতে হলে অপমান না করে নির্দিষ্ট কাজের...,13
5,মোবাইলগুলো কেড়ে নিয়ে ওই বেডিদের পিছনে লাঠি ভরে...,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,"সাথে নির্যাতনের কথা বলা খুবই বিপজ্জনক, তাই তা ...",13
6,ওইটা আমার বাপ আর তোর আম্মুর বোন রেন্ডির ছেলে,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,কারও জন্ম অপমান করা খুবই অপমানজনক ; এমন ভাষা ব...,13
7,এই মাদকাসক্ত পাটোয়ারীর পিছনে যেসব কথিত তাওহীদ...,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,কারও পরিবারকে এমন অপমান না করে ভুলটা পরিষ্কারভ...,11
8,এসব জানোয়ারদের মরন হয় না?,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...","কারও মৃত্যু কামনা করা বা, এমন কথা না বলে নির্দ...",13
9,তুই একটা পাঠান সৈনিক গো বীর্যে জম্ম সেটাই প্রম...,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,পরিবারকে টেনে গালি খুবই অমানবিক ।,6


# 22: Check generated length

In [ ]:
print(results_df["generated_length"].describe())

print(
    "Unique generated outputs:",
    results_df["generated_counterspeech"].nunique(),
    "/",
    len(results_df)
)

print(
    "Unique output rate:",
    results_df["generated_counterspeech"].nunique() / len(results_df)
)

results_df[
    [
        "reference_counterspeech",
        "generated_counterspeech",
        "generated_length"
    ]
].head(20)


count    302.000000
mean      10.082781
std        2.299018
min        5.000000
25%        8.250000
50%       10.000000
75%       12.000000
max       19.000000
Name: generated_length, dtype: float64
Unique generated outputs: 297 / 302
Unique output rate: 0.9834437086092715


,reference_counterspeech,generated_counterspeech,generated_length
0,এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,কারও পোশাক বা চেহারা এমন অপমান করা খুবই নিচু ম...,12
1,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,কোনো গোষ্ঠীর বিরুদ্ধে এমন গালি না দিয়ে নির্দি...,12
2,"মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",এভাবে কাউকে খারাপ দেখালে সমস্যার সমাধান হয় না ।,9
3,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,কাউকে লুচ্চা বলা বা পাগল বলে আক্রমণ না করে আচর...,13
4,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,ধর্ম কথা বলতে হলে অপমান না করে নির্দিষ্ট কাজের...,13
5,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,"সাথে নির্যাতনের কথা বলা খুবই বিপজ্জনক, তাই তা ...",13
6,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,কারও জন্ম অপমান করা খুবই অপমানজনক ; এমন ভাষা ব...,13
7,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,কারও পরিবারকে এমন অপমান না করে ভুলটা পরিষ্কারভ...,11
8,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...","কারও মৃত্যু কামনা করা বা, এমন কথা না বলে নির্দ...",13
9,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,পরিবারকে টেনে গালি খুবই অমানবিক ।,6


# 23: Save predictions

In [ ]:
prediction_csv = "/content/banglabert_encoder_decoder_test_predictions.csv"

results_df.to_csv(
    prediction_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved predictions:", prediction_csv)
files.download(prediction_csv)


Saved predictions: /content/banglabert_encoder_decoder_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 24: Metric helper functions

In [ ]:
smooth = SmoothingFunction().method1

def tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def calculate_bleu_scores(predictions, references):
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_3_scores = []
    bleu_4_scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            bleu_1_scores.append(0)
            bleu_2_scores.append(0)
            bleu_3_scores.append(0)
            bleu_4_scores.append(0)
            continue

        bleu_1_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smooth
        ))

        bleu_2_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth
        ))

        bleu_3_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1/3, 1/3, 1/3, 0),
            smoothing_function=smooth
        ))

        bleu_4_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth
        ))

    return {
        "BLEU-1": float(np.mean(bleu_1_scores)),
        "BLEU-2": float(np.mean(bleu_2_scores)),
        "BLEU-3": float(np.mean(bleu_3_scores)),
        "BLEU-4": float(np.mean(bleu_4_scores))
    }


def calculate_meteor(predictions, references):
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
            continue

        try:
            score = meteor_score([ref_tokens], pred_tokens)
        except:
            score = 0

        scores.append(score)

    return float(np.mean(scores))


def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))


def calculate_diversity(predictions):
    all_unigrams = []
    all_bigrams = []

    for text in predictions:
        tokens = tokenize_bn(text)
        all_unigrams.extend(tokens)
        all_bigrams.extend(get_ngrams(tokens, 2))

    distinct_1 = len(set(all_unigrams)) / len(all_unigrams) if len(all_unigrams) > 0 else 0
    distinct_2 = len(set(all_bigrams)) / len(all_bigrams) if len(all_bigrams) > 0 else 0

    diversity = (distinct_1 + distinct_2) / 2

    return {
        "Distinct-1": distinct_1,
        "Distinct-2": distinct_2,
        "Diversity": diversity
    }


# ============================================================
# Improved Novelty Calculation
# Old method: exact copy only
# New method: similarity-based novelty
# Novelty = 1 - Average Maximum Similarity with Training Targets
# ============================================================

def normalize_bn_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_jaccard_similarity(a, b):
    a_tokens = set(tokenize_bn(normalize_bn_text(a)))
    b_tokens = set(tokenize_bn(normalize_bn_text(b)))

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    intersection = len(a_tokens.intersection(b_tokens))
    union = len(a_tokens.union(b_tokens))

    return intersection / union if union > 0 else 0.0


def calculate_novelty(predictions, train_targets, similarity_threshold=0.70):
    train_targets = [normalize_bn_text(x) for x in train_targets]
    predictions = [normalize_bn_text(x) for x in predictions]

    max_similarities = []
    near_copy_count = 0

    for pred in predictions:
        if len(pred.strip()) == 0:
            max_similarities.append(0.0)
            continue

        similarities = [
            token_jaccard_similarity(pred, train_text)
            for train_text in train_targets
        ]

        max_similarity = max(similarities)
        max_similarities.append(max_similarity)

        if max_similarity >= similarity_threshold:
            near_copy_count += 1

    average_max_similarity = float(np.mean(max_similarities)) if len(max_similarities) > 0 else 0.0
    near_copy_rate = near_copy_count / len(predictions) if len(predictions) > 0 else 0.0

    novelty = 1 - average_max_similarity

    return {
        "Novelty": novelty,
        "Average Max Train Similarity": average_max_similarity,
        "Near Copy Rate": near_copy_rate
    }


bangla_abuse_words = [
    "চুদ",
    "মাগী",
    "হারামি",
    "শালা",
    "কুত্তা",
    "বাল",
    "খানকি",
    "বেশ্যা",
    "মাদারচোদ",
    "বোকাচোদা"
]


def calculate_abuse_word_rate(predictions, abuse_words):
    count = 0

    for text in predictions:
        text = str(text)
        found = any(word in text for word in abuse_words)

        if found:
            count += 1

    abuse_rate = count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Abuse-word rate": abuse_rate,
        "Abusive outputs": count
    }


def calculate_length_stats(predictions):
    lengths = [len(tokenize_bn(text)) for text in predictions]

    empty_count = sum(1 for text in predictions if len(str(text).strip()) == 0)
    empty_rate = empty_count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Average Generated Length": float(np.mean(lengths)),
        "Empty Output Rate": empty_rate
    }

# 25: Bengali ROUGE

In [ ]:
def simple_tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def get_ngrams_list(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


def rouge_n_score(pred, ref, n=1):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    pred_ngrams = get_ngrams_list(pred_tokens, n)
    ref_ngrams = get_ngrams_list(ref_tokens, n)

    if len(pred_ngrams) == 0 or len(ref_ngrams) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for ng in pred_ngrams:
        pred_counts[ng] = pred_counts.get(ng, 0) + 1

    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1

    overlap = 0

    for ng in ref_counts:
        overlap += min(ref_counts[ng], pred_counts.get(ng, 0))

    recall = overlap / len(ref_ngrams)
    precision = overlap / len(pred_ngrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(pred_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def calculate_custom_rouge(predictions, references):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        rouge1_scores.append(rouge_n_score(pred, ref, n=1))
        rouge2_scores.append(rouge_n_score(pred, ref, n=2))
        rougeL_scores.append(rouge_l_score(pred, ref))

    return {
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    }


rouge_scores = calculate_custom_rouge(predictions, references)

rouge_scores

{'rouge1': 0.07900674595762724,
 'rouge2': 0.013177205627585783,
 'rougeL': 0.07621588339448304}

# 26: BERTScore

In [ ]:
!pip install -q bert_score

from bert_score import score as bert_score

clean_predictions = [
    str(x).strip() if str(x).strip() else "ফাঁকা"
    for x in predictions
]

clean_references = [
    str(x).strip() if str(x).strip() else "ফাঁকা"
    for x in references
]

try:
    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="xlm-roberta-base",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )
except Exception as error:
    print("XLM-R BERTScore failed:", error)
    print("Using bert-base-multilingual-cased as fallback.")

    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="bert-base-multilingual-cased",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )

bertscore_precision = float(P.mean().item())
bertscore_recall = float(R.mean().item())
bertscore_f1 = float(F1.mean().item())

print("BERTScore Precision:", bertscore_precision)
print("BERTScore Recall:", bertscore_recall)
print("BERTScore F1:", bertscore_f1)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

calculating scores...
computing bert embedding.


  0%|          | 0/38 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/19 [00:00<?, ?it/s]

done in 1.03 seconds, 294.39 sentences/sec
BERTScore Precision: 0.8630061745643616
BERTScore Recall: 0.8542574644088745
BERTScore F1: 0.858546257019043


# 27: Calculate final metrics

In [ ]:
bleu_scores = calculate_bleu_scores(predictions, references)

meteor = calculate_meteor(predictions, references)

diversity_scores = calculate_diversity(predictions)

novelty_scores = calculate_novelty(
    predictions,
    train_df["target_text"].tolist(),
    similarity_threshold=0.70
)

abuse_scores = calculate_abuse_word_rate(
    predictions,
    bangla_abuse_words
)

length_scores = calculate_length_stats(predictions)

final_metrics = {
    "Model": "BanglaBERT",

    "BLEU-1": bleu_scores["BLEU-1"],
    "BLEU-2": bleu_scores["BLEU-2"],
    "BLEU-3": bleu_scores["BLEU-3"],
    "BLEU-4": bleu_scores["BLEU-4"],

    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],

    "BERTScore Precision": bertscore_precision,
    "BERTScore Recall": bertscore_recall,
    "BERTScore F1": bertscore_f1,

    "Diversity": diversity_scores["Diversity"],
    "Distinct-1": diversity_scores["Distinct-1"],
    "Distinct-2": diversity_scores["Distinct-2"],

    "Novelty": novelty_scores["Novelty"],
    "Average Max Train Similarity": novelty_scores["Average Max Train Similarity"],
    "Near Copy Rate": novelty_scores["Near Copy Rate"],

    "Abuse-word rate": abuse_scores["Abuse-word rate"],
    "Abusive outputs": abuse_scores["Abusive outputs"],

    "METEOR": meteor,

    "Average Generated Length": length_scores["Average Generated Length"],
    "Empty Output Rate": length_scores["Empty Output Rate"]
}

metrics_df = pd.DataFrame([final_metrics])

metrics_df


,Model,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore Precision,BERTScore Recall,...,Distinct-1,Distinct-2,Novelty,Average Max Train Similarity,Near Copy Rate,Abuse-word rate,Abusive outputs,METEOR,Average Generated Length,Empty Output Rate
0,BanglaBERT,0.066529,0.027853,0.017805,0.013658,0.079007,0.013177,0.076216,0.863006,0.854257,...,0.141544,0.444039,0.611315,0.388685,0.009934,0.006623,2,0.047291,10.082781,0.0


# 28: Save metrics

In [ ]:
metrics_csv = "/content/banglabert_encoder_decoder_final_metrics.csv"

metrics_df.to_csv(
    metrics_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved metrics:", metrics_csv)
files.download(metrics_csv)


Saved metrics: /content/banglabert_encoder_decoder_final_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 29: Zip and download model

In [ ]:
!zip -r /content/final_banglabert_encoder_decoder_counterspeech_v2.zip /content/final_banglabert_encoder_decoder_counterspeech_v2

files.download("/content/final_banglabert_encoder_decoder_counterspeech_v2.zip")


  adding: content/final_banglabert_encoder_decoder_counterspeech_v2/ (stored 0%)
  adding: content/final_banglabert_encoder_decoder_counterspeech_v2/generation_config.json (deflated 41%)
  adding: content/final_banglabert_encoder_decoder_counterspeech_v2/tokenizer_config.json (deflated 50%)
  adding: content/final_banglabert_encoder_decoder_counterspeech_v2/training_args.bin (deflated 53%)
  adding: content/final_banglabert_encoder_decoder_counterspeech_v2/config.json (deflated 76%)
  adding: content/final_banglabert_encoder_decoder_counterspeech_v2/model.safetensors (deflated 7%)
  adding: content/final_banglabert_encoder_decoder_counterspeech_v2/tokenizer.json (deflated 76%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>